# TP 10 – Les logs en temps réel (35 min)

L'application de réservation écrit ses logs dans `stream/logs/`. Une panne survient quelque part dans la matinée.

**Objectif** : la détecter en temps réel.

**Réussi si** : une alerte (taux d'erreur > 10 %) apparaît sur le bon endpoint, aux bonnes minutes.

**Avant de commencer**, dans un terminal à la racine du dépôt :

```bash
python scripts/stream_simulator.py logs --reset          # en direct : un fichier toutes les 2 secondes
# ou, si vous êtes en retard / en mode dégradé :
python scripts/stream_simulator.py logs --all --reset    # tout d'un coup
```

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")     # petit volume : 200 partitions par défaut, c'est trop

### 1. Lire les logs en continu (`readStream.text`) : une colonne `value` par ligne

Exemple de ligne :
```
2024-01-15T08:22:03.120 ERROR endpoint=/payment status=503 latency_ms=812 zone=161
```

In [ ]:
brut = ...

### 2. Parser avec `regexp_extract`

Motif fourni :
```python
motif = r"^(\S+) +(\w+) +endpoint=(\S+) status=(\d+) latency_ms=(\d+) zone=(\d+)$"
```
Colonnes : `ts` (`to_timestamp(..., "yyyy-MM-dd'T'HH:mm:ss.SSS")`), `niveau`, `endpoint`, `status` (int), `latence_ms` (int).

In [ ]:
motif = r"^(\S+) +(\w+) +endpoint=(\S+) status=(\d+) latency_ms=(\d+) zone=(\d+)$"
logs = brut.select(
    ...
)

### 3. Watermark de 2 minutes, fenêtre de 1 minute par `endpoint`

Calculer : `requetes` (count), `erreurs` (status >= 500), `taux_erreur`, `latence_p95` (`percentile_approx(latence_ms, 0.95)`).

In [ ]:
par_minute = (logs
    .withWatermark(...)
    .groupBy(...)
    .agg(...))

### 4. Écrire en mode **append** vers Parquet (`out/tp10/par_minute`), avec un checkpoint

En direct : `trigger(processingTime="5 seconds")` et `awaitTermination(120)`. Avec `--all` : `trigger(availableNow=True)`.

In [ ]:
q = ...

### 5. Lire le Parquet (en batch) et afficher les alertes : taux d'erreur > 10 %

Quel endpoint ? À quelles minutes ? Que devient la latence p95 ?

In [ ]:
...

### 6. Combien de lignes ont été ignorées par le watermark ?

Indice : `q.recentProgress`, champ `stateOperators[0]["numRowsDroppedByWatermark"]`.

Pourquoi les toutes dernières minutes n'apparaissent-elles pas dans le Parquet ?

In [ ]:
...

### Bonus – Fenêtre glissante de 5 minutes, toutes les minutes : l'alerte est-elle plus stable ?

In [ ]:
...